<img src='https://colab.research.google.com/assets/colab-badge.svg'>

# 🧠 MiniLLM v2 — DevLab
> Transformer decoder-only optimisé : **RoPE + RMSNorm + SwiGLU + GQA + Flash Attention**  
> Scalable de 15M → 1B paramètres — Entraînable sur GPU Colab gratuit (T4)

---

## 📋 Plan
| Étape | Description |
|-------|-------------|
| **1. Setup** | Installation des dépendances + clonage du repo |
| **2. Inspection** | Vérifier l'architecture avant d'entraîner |
| **3. Données** | Préparer ton corpus texte |
| **4. Entraînement** | Lancer et monitorer |
| **5. Génération** | Générer du texte avec le modèle entraîné |

> ⚡ **GPU requis** — Menu `Exécution > Modifier le type d'exécution > GPU (T4)`

---
## 1️⃣ Setup


In [ ]:
# @title ⚙️ Vérifier le GPU disponible
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f'✅ GPU détecté : {gpu_info}')
else:
    print('⚠️  Aucun GPU détecté !')
    print('   → Menu : Exécution > Modifier le type d\'exécution > GPU')

import torch
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA dispo: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM      : {vram:.1f} GB')

In [ ]:
# @title 📦 Installer les dépendances
!pip install tiktoken -q
print('✅ tiktoken installé')
import tiktoken, numpy, torch
print(f'tiktoken {tiktoken.__version__} | numpy {numpy.__version__} | torch {torch.__version__}')

In [ ]:
# @title 📂 Cloner le repo MiniLLM v2 depuis GitHub
import os

GITHUB_USER = 'bono-p'        # @param {type: 'string'}
REPO_NAME   = 'minillm_v2'   # @param {type: 'string'}

repo_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
if os.path.exists(REPO_NAME):
    print(f'Repo déjà cloné, mise à jour...')
    os.chdir(REPO_NAME)
    os.system('git pull')
else:
    print(f'Clonage depuis {repo_url}...')
    ret = os.system(f'git clone {repo_url}')
    if ret != 0:
        print('❌ Erreur de clonage. Vérifier que le repo est public.')
    else:
        os.chdir(REPO_NAME)
        print('✅ Repo cloné avec succès')

import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'\nRépertoire courant : {os.getcwd()}')
print('Fichiers :', sorted(os.listdir('.')))

In [ ]:
# @title 💾 (Optionnel) Monter Google Drive pour sauvegarder les checkpoints
MOUNT_DRIVE = False  # @param {type: 'boolean'}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/minillm_v2/checkpoints'
    import os; os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f'✅ Drive monté — checkpoints dans : {CHECKPOINT_DIR}')
else:
    CHECKPOINT_DIR = 'checkpoints'
    print(f'Checkpoints locaux (perdus à la fin de session) : {CHECKPOINT_DIR}')
    print('  → Activer MOUNT_DRIVE=True pour les sauvegarder sur Drive')

---
## 2️⃣ Inspection du modèle
> Vérifier l'architecture et les tailles disponibles avant d'entraîner.


In [ ]:
# @title 📊 Tableau comparatif de tous les presets
from config import PRESETS, ModelConfig

print(f'{'─'*68}')
print(f"  {'Size':>6} | {'Params':>8} | {'Layers':>6} | {'d_model':>7} | "
      f"{'Heads':>5} | {'KV':>3} | {'FFN':>5} | {'Ctx':>5}")
print(f'{'─'*68}')
for name, cfg in PRESETS.items():
    n    = cfg.count_params()
    val  = n/1e6
    attn = 'MHA' if cfg.kv_heads == cfg.n_heads else f'GQA×{cfg.n_heads//cfg.kv_heads}'
    print(f'  {name:>6} | {val:>6.1f}M | {cfg.n_layers:>6} | '
          f'{cfg.d_model:>7} | {cfg.n_heads:>5} | {cfg.kv_heads:>3} | '
          f'{cfg.ffn_hidden:>5} | {cfg.max_seq_len:>5}  ({attn})')
print(f'{'─'*68}')

In [ ]:
# @title 🔬 Inspecter un modèle en détail + test forward pass
import torch
from config import PRESETS
from model  import MiniLLM

MODEL_SIZE = '50M'  # @param ['15M', '50M', '125M', '350M', '1B']

cfg   = PRESETS[MODEL_SIZE]
model = MiniLLM(cfg)
n     = model.n_params

print(f'{'═'*55}')
print(f'  MiniLLM-{MODEL_SIZE}')
print(f'{'═'*55}')
print(f'  Paramètres total  : {n/1e6:.2f}M')

n_emb   = cfg.vocab_size * cfg.d_model
n_blks  = sum(p.numel() for nm, p in model.named_parameters() if 'blocks' in nm)
print(f'  ├── Embedding     : {n_emb/1e6:.2f}M  ({n_emb/n*100:.0f}%)')
print(f'  ├── Blocs (×{cfg.n_layers})   : {n_blks/1e6:.2f}M  ({n_blks/n*100:.0f}%)')
print(f'  └── LM Head       : {"partagé (tied)" if cfg.tie_embeddings else "séparé"}')
print()
print(f'  Mémoire estimée :')
print(f'  ├── Inférence bf16 : {n*2/1e9:.2f} GB')
print(f'  └── Entraînement  : ~{n*4*4/1e9:.2f} GB (poids + grads + Adam)')
print()

# Test forward
x = torch.randint(0, cfg.vocab_size, (2, 64))
with torch.no_grad():
    logits, loss = model(x, x)

expected = torch.log(torch.tensor(float(cfg.vocab_size))).item()
print(f'  Test forward pass :')
print(f'  ✅ Input  : {list(x.shape)}')
print(f'  ✅ Logits : {list(logits.shape)}')
print(f'  ✅ Loss   : {loss.item():.3f}  (attendu ≈ {expected:.2f} = log({cfg.vocab_size}))')
print(f'{'═'*55}')

---
## 3️⃣ Préparation des données
> Choisis une source de corpus ci-dessous. **Option A** = ton propre texte. **Option B** = corpus de démo téléchargé automatiquement.


In [ ]:
# @title 📤 Option A — Uploader ton propre fichier texte (.txt)
OPTION = 'B_demo'  # @param ['A_upload', 'B_demo']

import os, urllib.request

if OPTION == 'A_upload':
    from google.colab import files
    print('Sélectionne ton fichier .txt...')
    uploaded = files.upload()
    CORPUS_FILE = list(uploaded.keys())[0]
    size_mb = os.path.getsize(CORPUS_FILE) / 1e6
    print(f'✅ Fichier chargé : {CORPUS_FILE}  ({size_mb:.1f} MB)')

else:  # Option B — corpus de démo (extrait Wikisource FR ~5MB)
    CORPUS_FILE = 'corpus_demo.txt'
    if not os.path.exists(CORPUS_FILE):
        print('Téléchargement du corpus de démo (Wikisource FR)...')
        # Textes libres de droit — Projet Gutenberg français
        urls = [
            ('https://www.gutenberg.org/files/13951/13951-0.txt', 'hugo_miserables.txt'),
            ('https://www.gutenberg.org/files/4650/4650-0.txt',   'zola_germinal.txt'),
            ('https://www.gutenberg.org/files/5097/5097-0.txt',   'flaubert_bovary.txt'),
        ]
        with open(CORPUS_FILE, 'w', encoding='utf-8') as out:
            for url, name in urls:
                try:
                    print(f'  → {name}...')
                    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                    with urllib.request.urlopen(req, timeout=30) as r:
                        text = r.read().decode('utf-8', errors='replace')
                    out.write(text + '\n\n')
                    print(f'     {len(text)/1e6:.1f} MB')
                except Exception as e:
                    print(f'     ⚠ Erreur : {e}')

    size_mb = os.path.getsize(CORPUS_FILE) / 1e6
    print(f'\n✅ Corpus prêt : {CORPUS_FILE}  ({size_mb:.1f} MB)')
    with open(CORPUS_FILE, encoding='utf-8', errors='replace') as f:
        preview = f.read(300)
    print(f'\nAperçu :\n{preview}...')

In [ ]:
# @title 🔡 Tokeniser et préparer train/val
import os, sys
sys.path.insert(0, '.')
from prepare_data import prepare

VAL_RATIO = 0.01  # @param {type: 'slider', min: 0.005, max: 0.1, step: 0.005}

prepare(
    input_path = CORPUS_FILE,
    output_dir = 'data',
    val_ratio  = VAL_RATIO,
    encoding   = 'cl100k_base',
)

# Vérification rapide
import numpy as np
train = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val   = np.memmap('data/val.bin',   dtype=np.uint16, mode='r')
print(f'\n✅ Train : {len(train):,} tokens  |  Val : {len(val):,} tokens')

---
## 4️⃣ Entraînement
> Configure et lance l'entraînement. Les checkpoints sont sauvegardés automatiquement.


In [ ]:
# @title ⚙️ Configuration de l'entraînement
import torch
from config import TrainConfig, PRESETS

# ── Paramètres à modifier ─────────────────────────────────────────────
MODEL_SIZE   = '50M'    # @param ['15M', '50M', '125M']
MAX_ITERS    = 5000     # @param {type: 'integer'}
BATCH_SIZE   = 8        # @param {type: 'slider', min: 1, max: 32, step: 1}
GRAD_ACCUM   = 4        # @param {type: 'slider', min: 1, max: 32, step: 1}
SEQ_LEN      = 512      # @param [256, 512, 1024, 2048]
LEARNING_RATE = 3e-4    # @param {type: 'number'}
USE_COMPILE  = True     # @param {type: 'boolean'}

cfg              = TrainConfig()
cfg.model_size   = MODEL_SIZE
cfg.max_iters    = MAX_ITERS
cfg.batch_size   = BATCH_SIZE
cfg.grad_accum   = GRAD_ACCUM
cfg.seq_len      = SEQ_LEN
cfg.lr           = LEARNING_RATE
cfg.compile      = USE_COMPILE
cfg.out_dir      = CHECKPOINT_DIR
cfg.warmup_iters = max(200, MAX_ITERS // 20)

mcfg = PRESETS[MODEL_SIZE]
mcfg.max_seq_len = SEQ_LEN

batch_eff    = BATCH_SIZE * GRAD_ACCUM
tokens_per_it = batch_eff * SEQ_LEN
total_tokens  = MAX_ITERS * tokens_per_it

print(f'{'═'*55}')
print(f'  Config entraînement')
print(f'{'═'*55}')
print(f'  Modèle         : MiniLLM-{MODEL_SIZE}  ({mcfg.count_params()/1e6:.1f}M params)')
print(f'  Itérations     : {MAX_ITERS:,}')
print(f'  Batch effectif : {BATCH_SIZE} × {GRAD_ACCUM} = {batch_eff}')
print(f'  Séquence       : {SEQ_LEN} tokens')
print(f'  Tokens/iter    : {tokens_per_it:,}')
print(f'  Total tokens   : {total_tokens/1e6:.1f}M')
print(f'  Learning rate  : {LEARNING_RATE}')
print(f'  torch.compile  : {USE_COMPILE}')
print(f'{'─'*55}')

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    required = mcfg.count_params() * 4 * 4 / 1e9  # estimé
    ok = '✅' if vram_gb > required else '⚠️'
    print(f'  VRAM dispo : {vram_gb:.1f} GB  |  Estimé requis : {required:.1f} GB  {ok}')

print(f'{'═'*55}')
print('\n  Lance la cellule suivante pour démarrer !')

In [ ]:
# @title 🚀 Lancer l'entraînement
# ⚠️ Ne pas oublier de configurer la cellule précédente d'abord !
import sys, os
sys.path.insert(0, '.')
from train import train

# Réduire la fréquence de log pour Colab
cfg.log_every  = 50
cfg.eval_every = 500
cfg.save_every = 1000

print('Démarrage de l\'entraînement...')
print('Les checkpoints sont sauvegardés dans :', cfg.out_dir)
print()
train(cfg)

In [ ]:
# @title 📈 Visualiser la courbe de loss
import matplotlib.pyplot as plt
import re, os

# Lire les logs (capture stdout ou lire un fichier log si implémenté)
# Pour l'instant, vérifier les checkpoints disponibles
ckpt_dir = CHECKPOINT_DIR
ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.pt')])

if ckpts:
    import torch
    losses = []
    iters  = []
    for f in ckpts:
        ckpt = torch.load(os.path.join(ckpt_dir, f), map_location='cpu', weights_only=False)
        if ckpt.get('val_loss'):
            losses.append(ckpt['val_loss'])
            iters.append(ckpt.get('iter', 0))

    if losses:
        plt.figure(figsize=(10, 4))
        plt.plot(iters, losses, 'b-o', linewidth=2, markersize=4)
        plt.xlabel('Itération')
        plt.ylabel('Val Loss')
        plt.title('MiniLLM v2 — Courbe de validation')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('training_curve.png', dpi=150)
        plt.show()
        print(f'Meilleure val_loss : {min(losses):.4f} (iter {iters[losses.index(min(losses))]})')
    else:
        print('Pas encore de val_loss enregistrée.')
else:
    print(f'Aucun checkpoint trouvé dans {ckpt_dir}')

---
## 5️⃣ Génération de texte
> Charger le meilleur modèle et générer du texte interactivement.


In [ ]:
# @title 📂 Charger le modèle entraîné
import os, sys, torch
sys.path.insert(0, '.')
from generate import load_model

CHECKPOINT = 'best'  # @param ['best', 'latest']

if CHECKPOINT == 'best':
    ckpt_path = os.path.join(CHECKPOINT_DIR, 'best.pt')
else:
    # Prendre le dernier checkpoint numéroté
    ckpts = sorted([f for f in os.listdir(CHECKPOINT_DIR)
                    if f.startswith('ckpt_') and f.endswith('.pt')])
    ckpt_path = os.path.join(CHECKPOINT_DIR, ckpts[-1]) if ckpts else \
                os.path.join(CHECKPOINT_DIR, 'best.pt')

assert os.path.exists(ckpt_path), f'Checkpoint introuvable : {ckpt_path}'
gen_model, gen_cfg, gen_device = load_model(ckpt_path)
print(f'✅ Modèle chargé sur {gen_device}')

In [ ]:
# @title ✍️ Générer du texte
import torch, tiktoken
from generate import generate

PROMPT         = 'Il était une fois'  # @param {type: 'string'}
MAX_NEW_TOKENS = 200                   # @param {type: 'slider', min: 50, max: 500, step: 50}
TEMPERATURE    = 0.8                   # @param {type: 'slider', min: 0.1, max: 2.0, step: 0.1}
TOP_K          = 50                    # @param {type: 'slider', min: 0, max: 200, step: 10}
TOP_P          = 0.95                  # @param {type: 'slider', min: 0.5, max: 1.0, step: 0.05}
N_SAMPLES      = 1                     # @param {type: 'slider', min: 1, max: 5, step: 1}

enc    = tiktoken.get_encoding('cl100k_base')
tokens = enc.encode(PROMPT)

print(f'Prompt : "{PROMPT}"')
print(f'{'─'*50}')

for i in range(N_SAMPLES):
    if N_SAMPLES > 1:
        print(f'\n── Sample {i+1} ──────────────────────────')
    out = generate(
        gen_model, tokens,
        max_new_tokens = MAX_NEW_TOKENS,
        temperature    = TEMPERATURE,
        top_k          = TOP_K,
        top_p          = TOP_P,
        device         = gen_device,
    )
    text = enc.decode(out)
    print(text)
print(f'\n{'─'*50}')
print(f'Tokens générés : {len(out) - len(tokens)}')

In [ ]:
# @title 💾 Télécharger le modèle entraîné (sur ton PC)
from google.colab import files
import os

ckpt_path = os.path.join(CHECKPOINT_DIR, 'best.pt')
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print(f'Téléchargement de {ckpt_path}  ({size_mb:.0f} MB)...')
    files.download(ckpt_path)
else:
    print(f'Fichier introuvable : {ckpt_path}')
    print('Lance d\'abord l\'entraînement (section 4).')

---
## 🎁 Bonus — Inférence depuis un checkpoint local
> Si tu as téléchargé `best.pt` et que tu veux le ré-uploader plus tard.


In [ ]:
# @title 📤 Uploader un checkpoint existant (best.pt)
from google.colab import files
import shutil, os

print('Sélectionne ton fichier best.pt...')
uploaded = files.upload()
fname    = list(uploaded.keys())[0]
dest     = os.path.join(CHECKPOINT_DIR, 'best.pt')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
shutil.move(fname, dest)
print(f'✅ Checkpoint uploadé vers {dest}')
print('Tu peux maintenant aller à la section 5 pour générer du texte.')